## Ein Pydantic-Model erstellen

Wie lässt sich festlegen, welche Daten ein Objekt haben muss, und wie wird das beim Erzeugen automatisch geprüft? Genau dafür dient ein Model: Es beschreibt die Daten (Namen und Typen der Fields), Pydantic übernimmt die Validierung.

In diesem Notebook definieren wir ein `Person`-Model, sehen uns an, was Pydantic daraus macht, und prüfen, wann Fehler auftreten und wann nicht.

**Danach ist klar:**
- wie ein Model mit `BaseModel` definiert wird,
- was Pydantic beim Erzeugen prüft und umwandelt,
- wann Fehler auftreten und wann nicht.

Voraussetzung: Grundkenntnisse zu Klassen und Type Hints in Python.

<details>
<summary>Erklärung anzeigen</summary>

**Was der Code macht:** `BaseModel` ist die Basisklasse aller Pydantic-Models.
</details>

In [ ]:
from pydantic import BaseModel

<details>
<summary>Erklärung anzeigen</summary>

**Was der Code macht:** `Person` erbt von `BaseModel`. Die Annotationen legen die Fields und ihre Typen fest, Pydantic baut daraus die Validierung. Ohne Default-Werte sind alle drei Fields Pflicht.
</details>

In [ ]:
class Person(BaseModel):
    first_name: str
    last_name: str
    age: int

<details>
<summary>Erklärung anzeigen</summary>

**Was der Code macht:** `Person(...)` prüft die Werte gegen die Annotationen und legt eine Instanz an. Models akzeptieren nur Keyword-Argumente.
</details>

In [ ]:
p = Person(first_name="Evariste", last_name="Galois", age=20)

<details>
<summary>Erklärung anzeigen</summary>

**Was der Code macht:** `str(p)` liefert die lesbare Darstellung: Feldnamen und Werte, ohne Klassennamen.
</details>

In [ ]:
str(p)

<details>
<summary>Erklärung anzeigen</summary>

**Was der Code macht:** `repr(p)` ergänzt den Klassennamen und sieht aus wie der Aufruf, der die Instanz erzeugt.
</details>

In [ ]:
repr(p)

Welche Daten ein Model beschreibt, steht nicht in der Instanz, sondern in der Klasse. Wie lässt sich das einsehen?

<details>
<summary>Erklärung anzeigen</summary>

**Was der Code macht:** `model_fields` enthält die Metadaten aller Fields (Annotation, `required=True`). Sie gehören zur Klasse, deshalb steht hier `type(p)`. Der Zugriff über die Instanz (`p.model_fields`) ist seit Pydantic 2.11 veraltet.
</details>

In [ ]:
type(p).model_fields

<details>
<summary>Erklärung anzeigen</summary>

**Was der Code macht:** Dasselbe Ergebnis, direkt über den Klassennamen.
</details>

In [ ]:
Person.model_fields

### Validierungsfehler

Was passiert, wenn Daten nicht zum Model passen? Pydantic löst dann eine `ValidationError` aus, statt ungültige Daten stillschweigend zu übernehmen.

<details>
<summary>Erklärung anzeigen</summary>

**Was der Code macht:** `ValidationError` ist die Exception, die Pydantic bei ungültigen Daten auslöst.
</details>

In [ ]:
from pydantic import ValidationError

**Vorhersage:** Im folgenden Aufruf fehlen zwei der drei Fields. Meldet Pydantic nur den ersten Fehler oder alle?

<details>
<summary>Erklärung anzeigen</summary>

**Was der Code macht:** Der Aufruf lässt `first_name` und `age` weg. Pydantic sammelt alle Fehler und meldet sie gemeinsam in einer `ValidationError`. Jeder Eintrag nennt Feld, Fehlertyp (`missing`) und Input.

Kurz gesagt: Ein Aufruf liefert alle Probleme auf einmal.
</details>

In [ ]:
try:
    Person(last_name="Galois")
except ValidationError as exc:
    print(exc)

### Abgeleitete Werte mit `@property`

Wie lässt sich ein Wert anbieten, der sich aus den Fields ableiten lässt, ohne ihn zusätzlich zu speichern? Eine `@property` berechnet ihn bei jedem Zugriff neu.

<details>
<summary>Erklärung anzeigen</summary>

**Was der Code macht:** Die neue Klasse erweitert `Person` um eine `@property`. `display_name` ist kein Field, sondern wird bei jedem Zugriff aus `first_name` und `last_name` berechnet und kann deshalb nicht abweichen.

Hinweis: Die Property erscheint weder in `model_fields` noch in `model_dump()`. Dafür gibt es `@computed_field`.
</details>

In [ ]:
class Person(BaseModel):
    first_name: str
    last_name: str
    age: int

    @property
    def display_name(self):
        return f"{self.first_name} {self.last_name}"

<details>
<summary>Erklärung anzeigen</summary>

**Was der Code macht:** Die Instanz wird neu erzeugt, weil `p` noch auf ein Objekt der alten Klasse zeigt.
</details>

In [ ]:
p = Person(first_name="Evariste", last_name="Galois", age=20)

<details>
<summary>Erklärung anzeigen</summary>

**Was der Code macht:** Der Zugriff ruft die Property ohne Klammern auf. Der Wert wird aus den aktuellen Fields zusammengesetzt.
</details>

In [ ]:
p.display_name

<details>
<summary>Erklärung anzeigen</summary>

**Was der Code macht:** Fields lassen sich wie Attribute lesen.
</details>

In [ ]:
p.first_name

<details>
<summary>Erklärung anzeigen</summary>

**Was der Code macht:** Der Wert ist ein `int`, wie annotiert.
</details>

In [ ]:
p.age

### Zuweisung und Validierung

Sind Models nach dem Erzeugen unveränderlich, und wird bei einer späteren Zuweisung erneut geprüft?

<details>
<summary>Erklärung anzeigen</summary>

**Was der Code macht:** `age` wird direkt an der Instanz geändert. Models sind standardmäßig veränderbar.
</details>

In [ ]:
p.age = 21

<details>
<summary>Erklärung anzeigen</summary>

**Was der Code macht:** `age=21`: Die bestehende Instanz wurde verändert, es gibt keine Kopie.
</details>

In [ ]:
p

<details>
<summary>Erklärung anzeigen</summary>

**Was der Code macht:** Der String `"twenty"` lässt sich nicht in `int` umwandeln (Coercion), Fehlertyp `int_parsing`. `"20"` wäre akzeptiert und zu `20` konvertiert worden.

Hinweis: Mit dem Strict Mode wird die Konvertierung abgeschaltet.
</details>

In [ ]:
try:
    Person(first_name="Evariste", last_name="Galois", age="twenty")
except ValidationError as exc:
    print(exc)

<details>
<summary>Erklärung anzeigen</summary>

**Was der Code macht:** `p` ist unverändert. Der fehlgeschlagene Aufruf hat die bestehende Instanz nicht berührt.
</details>

In [ ]:
p

<details>
<summary>Erklärung anzeigen</summary>

**Was der Code macht:** Gültige Zuweisung: `age` ist wieder `20`.
</details>

In [ ]:
p.age = 20

**Vorhersage:** Jetzt wird `"twenty"` einem bestehenden Objekt zugewiesen. Löst das einen Fehler aus? Und was steht danach in `p`?

<details>
<summary>Erklärung anzeigen</summary>

**Was der Code macht:** Keine Exception: Pydantic validiert standardmäßig nur beim Erzeugen. Bei Zuweisungen muss das mit `ConfigDict(validate_assignment=True)` eingeschaltet werden.
</details>

In [ ]:
p.age = "twenty"

<details>
<summary>Erklärung anzeigen</summary>

**Was der Code macht:** `p` zeigt `age='twenty'`, einen String im `int`-Field.

Kurz gesagt: Gültige Daten garantiert Pydantic beim Erzeugen, bei Zuweisungen nur mit `validate_assignment=True`.
</details>

In [ ]:
p

### Praxisbeispiel: Datensätze einlesen

Daten aus einer API oder Datei liegen meist als Dictionaries vor, oft mit kleinen Unsauberkeiten. Wie verarbeitet `Person` solche Einträge, und wie lassen sich fehlerhafte aussortieren?

**Vorhersage:** Der zweite Eintrag enthält das Alter als String `"41"`, der dritte hat keinen `last_name`. Welche Einträge werden akzeptiert, und welchen Typ hat `age` danach?

<details>
<summary>Erklärung anzeigen</summary>

**Was der Code macht:** Die Schleife erzeugt je Eintrag eine `Person`. `**entry` entpackt das Dictionary in Keyword-Argumente. Ungültige Einträge fängt `except ValidationError` ab, die Schleife läuft weiter. `exc.error_count()` liefert die Zahl der Fehler.

Der String `"41"` wird zu `41` umgewandelt (Coercion). Beim dritten Eintrag fehlt `last_name`.

Kurz gesagt: Das Model nimmt kleine Unsauberkeiten an und lehnt Unvollständiges ab.
</details>

In [ ]:
entries = [
    {"first_name": "Ada", "last_name": "Lovelace", "age": 36},
    {"first_name": "Alan", "last_name": "Turing", "age": "41"},
    {"first_name": "Grace", "age": 85},
]

for entry in entries:
    try:
        person = Person(**entry)
    except ValidationError as exc:
        print(f"Ungültiger Eintrag {entry}: {exc.error_count()} Fehler")
    else:
        print(f"{person.display_name}, {person.age} Jahre (Typ: {type(person.age).__name__})")

### Zusammenfassung

- `BaseModel` plus Annotationen definiert ein Model; alle Fields ohne Default sind Pflicht.
- `Person(...)` validiert beim Erzeugen, wandelt passende Werte um (`"41"` → `41`) und meldet alle Fehler gesammelt in einer `ValidationError`.
- `model_fields` gehört zur Klasse und wird über `Person.model_fields` oder `type(p).model_fields` gelesen.
- Eine `@property` liefert abgeleitete Werte, erscheint aber nicht in `model_fields` und `model_dump()`.
- Zuweisungen werden standardmäßig nicht validiert; dafür dient `validate_assignment=True`.